# Preflight for convergence protocol v9 (about 2 minutes, no training)

Run this once in Molab, in the folder that holds the helper `.py` files (including the two `test_*.py` files), before Part 1.
It checks, on this GPU:

1. **Stopping and resume logic**: early stopping, epoch cap, a session pause that never finalizes training,
   and a resumed run that reproduces an uninterrupted run exactly (`test_convergence_protocol.py`).
2. **GPU data path**: the on-GPU training patches, graph-prior projection and sliding-window inference give
   exactly the same values as the original CPU code (`test_gpu_equivalence.py`).
3. The GPU, free VRAM and `monai==1.5.1` (needed by Parts 3 and 5).

Marimo-safe: no underscore-prefixed names, each global defined once. No Hugging Face token needed.

In [ ]:
import os, sys, subprocess, time
from pathlib import Path

HELPER_DIR = Path(globals().get("__file__") or os.getcwd()).resolve()
HELPER_DIR = HELPER_DIR if HELPER_DIR.is_dir() else HELPER_DIR.parent
REQUIRED = ["brats_practical.py", "brats_experiments.py", "brats_gpu.py", "brats_protocol.py", "brats_graph.py",
            "brats_workflow.py", "brats_reporting.py", "brats_bundles.py", "brats_transfer.py",
            "test_convergence_protocol.py", "test_gpu_equivalence.py"]
MISSING = [name for name in REQUIRED if not (HELPER_DIR / name).exists()]
print("helper folder:", HELPER_DIR)
print("missing files:", MISSING or "none")

In [ ]:
import torch
GPU_REPORT = {"torch": torch.__version__, "cuda_available": torch.cuda.is_available()}
if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    GPU_REPORT.update(device=torch.cuda.get_device_name(0), total_gb=round(total_bytes / 1e9, 1),
                      free_gb=round(free_bytes / 1e9, 1), bf16=torch.cuda.is_bf16_supported())
try:
    import monai
    GPU_REPORT["monai"] = monai.__version__
except ImportError:
    GPU_REPORT["monai"] = "NOT INSTALLED (run: pip install --no-deps monai==1.5.1)"
print(GPU_REPORT)

In [ ]:
def run_test_file(relative):
    started = time.time()
    done = subprocess.run([sys.executable, str(HELPER_DIR / relative)], cwd=str(HELPER_DIR),
                          capture_output=True, text=True, timeout=1800)
    print("=" * 70, "\n" + relative, "(%.0f s, exit %d)" % (time.time() - started, done.returncode))
    print(done.stdout[-6000:])
    if done.returncode != 0:
        print(done.stderr[-6000:])
    return done.returncode == 0

TEST_RESULTS = {} if MISSING else {name: run_test_file(name) for name in
                                   ("test_convergence_protocol.py", "test_gpu_equivalence.py")}

In [ ]:
PREFLIGHT_OK = (not MISSING and TEST_RESULTS and all(TEST_RESULTS.values()) and GPU_REPORT["cuda_available"]
                and not str(GPU_REPORT.get("monai", "")).startswith("NOT"))
if PREFLIGHT_OK:
    print("PREFLIGHT PASSED: start Part 1 (part1_corrected).")
else:
    print("PREFLIGHT FAILED: do not start training. Send the output above.",
          {"missing": MISSING, "tests": TEST_RESULTS, "gpu": GPU_REPORT.get("cuda_available"), "monai": GPU_REPORT.get("monai")})